# Diet Optimization with cuOpt Python API

This notebook demonstrates how to solve the classic diet optimization problem using the cuOpt Python API. The problem involves selecting foods to meet nutritional requirements while minimizing cost.

## Problem Description

We need to select quantities of different foods to:
- Meet minimum and maximum nutritional requirements
- Minimize total cost
- Satisfy additional constraints (like limiting dairy servings)

The nutrition guidelines are based on USDA Dietary Guidelines for Americans, 2005.


## Environment Setup

First, let's check if we have a GPU available and install necessary dependencies.


In [1]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["amd-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

True

In [2]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12 rapids-logger==0.1.19
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13 rapids-logger==0.1.19

## Import Required Libraries


In [4]:
import numpy as np
import pandas as pd
from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings
import time


## Problem Data Setup

Define the nutrition guidelines, food costs, and nutritional values for each food item.


In [5]:
# Nutrition guidelines based on USDA Dietary Guidelines for Americans, 2005
# http://www.health.gov/DietaryGuidelines/dga2005/

# minimum and maximum values for each category
categories = {
    "calories": {
        "min": 1800,
        "max": 2200
    },
    "protein": {
        "min": 91,
        "max": float('inf')
    },
    "fat": {
        "min": 0,
        "max": 65
    },
    "sodium": {
        "min": 0,
        "max": 1779
    }
}



In [6]:
# Food costs per serving
food_costs = {
    "hamburger": 2.49,
    "chicken": 2.89,
    "hot dog": 1.50,
    "fries": 1.89,
    "macaroni": 2.09,
    "pizza": 1.99,
    "salad": 2.49,
    "milk": 0.89,
    "ice cream": 1.59
}

# Nutrition values for each food (per serving)
nutrition_data = {
    "hamburger": [410, 24, 26, 730],
    "chicken": [420, 32, 10, 1190],
    "hot dog": [560, 20, 32, 1800],
    "fries": [380, 4, 19, 270],
    "macaroni": [320, 12, 10, 930],
    "pizza": [320, 15, 12, 820],
    "salad": [320, 31, 12, 1230],
    "milk": [100, 8, 2.5, 125],
    "ice cream": [330, 8, 10, 180]
}


In [7]:
# Create a DataFrame for better visualization
nutrition_df = pd.DataFrame(nutrition_data, index=categories.keys()).T
nutrition_df.columns = [f"{cat} (per serving)" for cat in categories.keys()]
print("Nutritional Values per Serving:")
print(nutrition_df)

Nutritional Values per Serving:
           calories (per serving)  protein (per serving)  fat (per serving)  \
hamburger                   410.0                   24.0               26.0   
chicken                     420.0                   32.0               10.0   
hot dog                     560.0                   20.0               32.0   
fries                       380.0                    4.0               19.0   
macaroni                    320.0                   12.0               10.0   
pizza                       320.0                   15.0               12.0   
salad                       320.0                   31.0               12.0   
milk                        100.0                    8.0                2.5   
ice cream                   330.0                    8.0               10.0   

           sodium (per serving)  
hamburger                 730.0  
chicken                  1190.0  
hot dog                  1800.0  
fries                     270.0  
macaron

## Problem Formulation

Now we'll create the optimization problem using the cuOpt Python API as MILP. The problem has:
- **Variables**: Amount of each food to buy (continuous, non-negative)
- **Objective**: Minimize total cost
- **Constraints**: Meet nutritional requirements (minimum and maximum bounds)

Since these are price per serving, you need to have a whole number for number of product that will be used.


In [8]:
# Create the optimization problem
problem = Problem("diet_optimization")

# Add decision variables for each food (amount to buy)
buy_vars = {}
for food_name in food_costs:
    # Using integer type for amount of food to buy since serving needs to be whole number
    # And this converts the problem to MILP
    var = problem.addVariable(name=f"{food_name}", vtype=VType.INTEGER, lb=0.0, ub=float('inf'))
    buy_vars[food_name] = var

print(f"Created {len(buy_vars)} decision variables for foods")
print(f"Variables: {[var.getVariableName() for var in buy_vars.values()]}")


Created 9 decision variables for foods
Variables: ['hamburger', 'chicken', 'hot dog', 'fries', 'macaroni', 'pizza', 'salad', 'milk', 'ice cream']


In [9]:
objective_expr = LinearExpression([], [], 0.0)

for var in buy_vars.values():
    if food_costs[var.getVariableName()] != 0:  # Only include non-zero coefficients
        objective_expr += var * food_costs[var.getVariableName()]

# Set objective function: minimize total cost
problem.setObjective(objective_expr, sense.MINIMIZE)


In [10]:
# Add nutrition constraints
constraint_names = []

for i, category in enumerate(categories):
    # Calculate total nutrition from all foods for this category
    nutrition_expr = LinearExpression([], [], 0.0)
    
    for food_name in food_costs: 
        nutrition_value = nutrition_data[food_name][i]
        if nutrition_value != 0:  # Only include non-zero coefficients
            nutrition_expr += buy_vars[food_name] * nutrition_value
    
    # Add constraint: min_nutrition[i] <= nutrition_expr <= max_nutrition[i]
    min_val = categories[category]["min"]
    max_val = categories[category]["max"]
    
    if max_val == float('inf'):
        # Only lower bound constraint
        constraint = problem.addConstraint(nutrition_expr >= min_val, name=f"min_{category}")
        constraint_names.append(f"min_{category}")
    else:
        # Range constraint (both lower and upper bounds)
        constraint = problem.addConstraint(nutrition_expr >= min_val, name=f"min_{category}")
        constraint_names.append(f"min_{category}")
        constraint = problem.addConstraint(nutrition_expr <= max_val, name=f"max_{category}")
        constraint_names.append(f"max_{category}")

print(f"Added {len(constraint_names)} nutrition constraints")
print(f"Constraints: {constraint_names}")


Added 7 nutrition constraints
Constraints: ['min_calories', 'max_calories', 'min_protein', 'min_fat', 'max_fat', 'min_sodium', 'max_sodium']


## Solver Configuration and Solution

Configure the solver settings and solve the optimization problem.


In [11]:
# Configure solver settings
settings = SolverSettings()
settings.set_parameter("time_limit", 60.0)  # 60 second time limit
settings.set_parameter("log_to_console", True)  # Enable solver logging
settings.set_parameter("method", 1)  # PDLP only (rocopt: avoid Concurrent dispatcher / Barrier deadlock)

print("Solver configured with 60-second time limit")


Solver configured with 60-second time limit


In [12]:
# Solve the problem
print("Solving diet optimization problem...")
print(f"Problem type: {'MIP' if problem.IsMIP else 'LP'}")

start_time = time.time()
problem.solve(settings)
solve_time = time.time() - start_time

print(f"\nSolve completed in {solve_time:.3f} seconds")
print(f"Solver status: {problem.Status.name}")
print(f"Objective value: ${problem.ObjValue:.2f}")


Solving diet optimization problem...
Problem type: MIP
Setting parameter time_limit to 6.000000e+01
Setting parameter log_to_console to true
Setting parameter method to 1
cuOpt version: 26.1.0, git hash: 7c418b6, host arch: x86_64, device archs: gfx942
CPU: AMD EPYC 9654 96-Core Processor, threads (physical/logical): 192/384, RAM: 307.78 GiB
HIP/ROCm 70152.80, device: AMD Instinct MI300X (ID 0), VRAM: 191.98 GiB
HIP device UUID: 62656132-6464-3130-3961-323936363137

Solving a problem with 7 constraints, 9 variables (9 integers), and 63 nonzeros
Problem scaling:
Objective coefficents range:          [9e-01, 3e+00]
Constraint matrix coefficients range: [2e+00, 2e+03]
Constraint rhs / bounds range:        [0e+00, 2e+03]
Variable bounds range:                [0e+00, 0e+00]

Original problem: 7 constraints, 9 variables, 63 nonzeros
Calling Papilo presolver
Presolve status: reduced the problem
Presolve removed: 3 constraints, 3 variables, 39 nonzeros
Presolved problem: 4 constraints, 6 varia

In [13]:
def print_solution():
    """Print the optimal solution in a readable format"""
    if problem.Status.name == "Optimal" or problem.Status.name == "FeasibleFound":
        print(f"\nOptimal Solution Found!")
        print(f"Total Cost: ${problem.ObjValue:.2f}")
        print("\nFood Purchases:")
        
        total_cost = 0
        for var in buy_vars.values():
            amount = var.getValue()
            if amount > 0.0001:  # Only show foods with significant amounts
                food_cost = amount * food_costs[var.getVariableName()]
                total_cost += food_cost
                print(f"  {var.getVariableName()}: {amount:.3f} servings (${food_cost:.2f})")
        
        print(f"\nTotal Cost: ${total_cost:.2f}")
        
        # Check nutritional intake
        print("\nNutritional Intake:")
        for i, category in enumerate(categories):
            total_nutrition = 0
            for var in buy_vars.values():
                amount = var.getValue()
                nutrition_value = nutrition_data[var.getVariableName()][i]
                total_nutrition += amount * nutrition_value
            
            min_req = categories[category]["min"]
            max_req = categories[category]["max"]
            
            # Check constraints with tolerance for floating point precision
            tolerance = 1e-6
            min_satisfied = total_nutrition >= (min_req - tolerance)
            max_satisfied = (max_req == float('inf')) or (total_nutrition <= (max_req + tolerance))
            status = "✓" if (min_satisfied and max_satisfied) else "✗"
            
            if max_req == float('inf'):
                print(f"  {category}: {total_nutrition:.1f} (min: {min_req}) {status}")
            else:
                print(f"  {category}: {total_nutrition:.1f} (min: {min_req}, max: {max_req}) {status}")
    else:
        print(f"No optimal solution found. Status: {problem.Status.name}")

print_solution()



Optimal Solution Found!
Total Cost: $12.78

Food Purchases:
  milk: 9.000 servings ($8.01)
  ice cream: 3.000 servings ($4.77)

Total Cost: $12.78

Nutritional Intake:
  calories: 1890.0 (min: 1800, max: 2200) ✓
  protein: 96.0 (min: 91) ✓
  fat: 52.5 (min: 0, max: 65) ✓
  sodium: 1665.0 (min: 0, max: 1779) ✓


## Adding Additional Constraints

Now let's demonstrate how to add additional constraints to the existing model. We'll add a constraint to limit dairy servings to at most 6.


In [14]:
# Create LinearExpression for dairy constraint
dairy_expr = buy_vars["milk"] + buy_vars["ice cream"]

dairy_constraint = problem.addConstraint(dairy_expr <= 6, name="limit_dairy")

In [ ]:
# Solve the problem again with the new constraint
print("\nSolving with dairy constraint...")
print(f"Problem now has {problem.NumVariables} variables and {problem.NumConstraints} constraints")

start_time = time.time()
problem.solve(settings)
solve_time = time.time() - start_time

print(f"\nSolve completed in {solve_time:.3f} seconds")
print(f"Solver status: {problem.Status.name}")
print(f"Objective value: ${problem.ObjValue:.2f}")


## Solution Comparison

Let's compare the solutions before and after adding the dairy constraint to see the impact.


In [ ]:
# Display the new solution
print_solution()

## Conclusion

This notebook demonstrated how to:

1. **Formulate a diet optimization problem** using the cuOpt Python API
2. **Set up decision variables** for food quantities
3. **Define an objective function** to minimize total cost
4. **Add nutritional constraints** with both lower and upper bounds
5. **Solve the optimization problem** using cuOpt's high-performance solver
6. **Add additional constraints** to the existing model
7. **Analyze and compare solutions** before and after constraint modifications

The cuOpt Python API provides a clean, intuitive interface for building and solving optimization problems, making it easy to model complex real-world scenarios like diet optimization.

### Key Benefits of cuOpt:
- **High Performance**: GPU-accelerated solving for large-scale problems
- **Easy to Use**: Intuitive Python API similar to other optimization libraries
- **Flexible**: Support for both LP and MIP problems
- **Scalable**: Handles problems with thousands of variables and constraints efficiently



SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.